# Event Catering Preparation Under Demand Uncertainty
### Monte Carlo + next-event discrete-event simulation of how much food to prepare

**Summary.** Caterers prepare food from advance bookings, but real attendance and per-guest consumption are random. Too little food causes shortages; too much wastes money. This notebook models attendance (Binomial), per-guest consumption (discrete distribution) and the buffet itself (next-event queueing simulation with inventory depletion), then runs **10,000 Monte Carlo replications per scenario** to find the preparation quantity $Q$ that minimises expected cost subject to a **shortage probability of at most 5%**. The simulator is verified against analytic results (Binomial moments, M/M/1 and Erlang-C queue waits, a hand-solved optimisation case, the newsvendor quantile), and a sensitivity study, a $N^{-1/2}$ convergence study and a multi-category (rice / meat / vegetables) extension are included.

> **All cost, consumption, arrival and service parameters are ASSUMED** (no measured catering records were used). Sensitivity analysis in §5 shows how much each assumption matters. Random seed: `SEED = 42` throughout.

In [1]:
# Setup (Colab: packages are preinstalled; this keeps the notebook reproducible elsewhere)
!pip install -q numpy scipy matplotlib pandas

import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from math import factorial

SEED = 42                                   # master seed (reproducibility)
os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "font.size": 10})
T0 = time.time()

## 1. Problem and question

Event caterers in Sri Lanka typically cook against advance guest bookings. On the day, the number who attend and how many portions each guest takes are uncertain. Shortages hurt customer satisfaction; surplus wastes ingredients (the food-service sector is a major source of food waste [5]).

**Question.** For an event with $N$ invited guests (100 to 500), *how many portions $Q$ should be prepared to minimise expected total cost while keeping the probability of any food shortage at or below 5%?* And how sensitive is that answer to attendance rate, consumption variability and the cost structure?

## 2. Model, assumptions and equations

**Assumptions** (all parameters assumed; see §5 sensitivity)
1. Each invited guest attends independently with probability $p$.
2. Each attending guest takes $X$ portions, i.i.d., from a discrete distribution on $\{0,1,2,3\}$ (mean 1.5).
3. Guests arrive as a Poisson stream (rate $\lambda$) truncated after the $A$ attendees; $c$ servers at the buffet, exponential service (rate $\mu$ each), FIFO queue.
4. Food is taken when a guest's service completes; if stock runs short the guest receives what remains (partial service) and the rest is *unmet demand*.
5. Preparation is fixed before the event (single period, no re-cooking). Costs: $c_\text{prep}$ per prepared portion, $c_\text{short}$ per unmet portion (LKR).

**Input modelling and random variate generation**

$$A\sim\text{Binomial}(N,p),\qquad E[A]=Np,\qquad D=\sum_{i=1}^{A}X_i .$$

Given $A$, the numbers of guests taking each portion size are $\text{Multinomial}(A,\boldsymbol\pi)$, so $D=\sum_k v_k n_k$ is generated in one vectorised draw. By the law of total variance
$$E[D]=Np\,\mu_X,\qquad \operatorname{Var}(D)=Np\,\sigma_X^2+Np(1-p)\,\mu_X^2 .$$
Inter-arrival times use inverse-transform sampling: $T=-\ln(U)/\lambda$, $U\sim U[0,1]$.

**Cost and decision problem**
$$C(Q)=c_\text{prep}\,Q+c_\text{short}\,(D-Q)^+,\qquad Q^\*=\arg\min_Q E[C(Q)]\ \ \text{s.t.}\ \ P(D>Q)\le\alpha=0.05 .$$
Unconstrained, a one-unit increase in $Q$ changes expected cost by $c_\text{prep}-c_\text{short}P(D>Q)$, so the optimum is the smallest $Q$ with
$$F_D(Q)\ \ge\ 1-\frac{c_\text{prep}}{c_\text{short}}\quad(\text{newsvendor critical fractile}).$$

**Monte Carlo estimation (law of large numbers)**
$$\hat C=\frac1n\sum_{j=1}^{n}C_j,\qquad \text{95\% CI}=\hat C\pm1.96\,\frac{s}{\sqrt n},\qquad \text{error}\propto n^{-1/2}.$$
All candidate $Q$ are evaluated on the *same* $n$ demand draws (common random numbers), which removes noise from comparisons between quantities.

**Discrete-event simulation (next-event time advance).** State: clock, busy servers, FIFO queue, inventory. Future-event list (binary heap) holds ARRIVAL and DEPARTURE events; the clock jumps to the earliest event. Ties are broken deterministically (departures before arrivals, then insertion order). For queue verification with $\rho=\lambda/(c\mu)<1$:
$$W_q^{M/M/1}=\frac{\lambda}{\mu(\mu-\lambda)},\qquad W_q^{M/M/c}=\frac{C(c,a)}{c\mu-\lambda},\quad C(c,a)=\frac{\frac{a^c}{c!}\frac1{1-\rho}}{\sum_{k=0}^{c-1}\frac{a^k}{k!}+\frac{a^c}{c!}\frac1{1-\rho}},\ a=\frac\lambda\mu .$$

| Component | Concept applied |
|---|---|
| $A$, $X$, inter-arrival and service draws | Input modelling and random variate generation |
| $\hat C$, CIs, shortage probability, $n^{-1/2}$ study | Monte Carlo integration and the law of large numbers |
| Buffet queue, event list, inventory depletion, M/M/c checks | Discrete-event simulation and queueing |

## 3. Implementation

The model is split into: parameters and input models, a fast vectorised Monte Carlo layer (used for the grid search), the next-event DES layer, and a multi-category extension.

In [2]:
# 3.0  Imports and conventions
from dataclasses import dataclass, field, replace
import heapq
import numpy as np

print('Model code loaded')

Model code loaded


In [3]:
# 3.  Parameters
@dataclass(frozen=True)
class Params:
    N: int = 200                      # invited guests
    p: float = 0.90                   # P(an invited guest attends)
    values: tuple = (0, 1, 2, 3)      # portions a guest may take
    probs: tuple = (0.05, 0.50, 0.35, 0.10)   # their probabilities (mean 1.5)
    c_prep: float = 300.0             # LKR per prepared portion
    c_short: float = 1500.0           # LKR penalty per unmet portion
    lam: float = 5.0                  # guest arrival rate (guests/min)
    mu: float = 1.5                   # service rate per server (guests/min)
    servers: int = 4                  # buffet service points

    @property
    def mean_portions(self):
        return float(np.dot(self.values, self.probs))

    @property
    def var_portions(self):
        v = np.array(self.values, float)
        return float(np.dot(v**2, self.probs) - self.mean_portions**2)

    @property
    def mean_demand(self):            # E[D] = N p E[X]  (Wald)
        return self.N * self.p * self.mean_portions

    @property
    def var_demand(self):             # Var D = Np E[X^2] ... via law of total variance
        EX, VX = self.mean_portions, self.var_portions
        return self.N * self.p * VX + self.N * self.p * (1 - self.p) * EX**2


In [4]:
# 3.  Input models (random variate generation)
def draw_attendance(par, rng, size):
    """A ~ Binomial(N, p)."""
    return rng.binomial(par.N, par.p, size=size)


def draw_demand(par, rng, size):
    """Total portions D = sum of X_i over the A attending guests.
    Given A, the counts of guests choosing each portion size are
    Multinomial(A, probs); D = counts . values.  Fully vectorised."""
    A = draw_attendance(par, rng, size)
    counts = rng.multinomial(A, par.probs)          # shape (size, len(values))
    return A, counts @ np.array(par.values)


In [5]:
# 3.  Fast layer: Monte Carlo + grid search with common random numbers
def ci95(x, axis=0):
    """Mean and 95% half-width (normal approx, 1.96 s/sqrt(n))."""
    n = x.shape[axis]
    m = x.mean(axis=axis)
    hw = 1.96 * x.std(axis=axis, ddof=1) / np.sqrt(n)
    return m, hw


def evaluate_grid(par, Qgrid, reps=10_000, seed=42):
    """Evaluate every Q in Qgrid on the SAME `reps` demand draws."""
    rng = np.random.default_rng(seed)
    _, D = draw_demand(par, rng, reps)
    Q = np.asarray(Qgrid)[None, :]
    D = D[:, None]
    unmet = np.maximum(D - Q, 0)
    left = np.maximum(Q - D, 0)
    short = (unmet > 0).astype(float)
    cost = par.c_prep * Q + par.c_short * unmet
    out = {"Q": np.asarray(Qgrid)}
    for name, arr in [("cost", cost), ("unused", left), ("unmet", unmet), ("p_short", short)]:
        m, hw = ci95(arr)
        out[name], out[name + "_hw"] = m, hw
    return out


def optimise(par, Qgrid, reps=10_000, seed=42, alpha=0.05):
    """Min expected cost s.t. P(shortage) <= alpha.  Returns dict."""
    g = evaluate_grid(par, Qgrid, reps, seed)
    feas = g["p_short"] <= alpha
    if not feas.any():
        raise ValueError("No Q in grid satisfies the shortage constraint; widen the grid.")
    idx_f = np.where(feas)[0]
    i = idx_f[np.argmin(g["cost"][idx_f])]
    j = int(np.argmin(g["cost"]))                    # unconstrained optimum
    return {"grid": g, "i_opt": i, "Q_opt": int(g["Q"][i]), "i_unc": j,
            "Q_unc": int(g["Q"][j]), "constraint_binding": bool(i != j)}


In [6]:
# 3.  DES layer: next-event simulation with a future-event list (heapq)
ARRIVAL, DEPARTURE = 1, 0     # on equal times, departures are processed first


def simulate_event(par, Q, rng, n_guests=None, portions=None):
    """One catering event, next-event time advance.

    State: clock, servers busy, FIFO queue (arrival time, portions wanted),
           inventory.  Event list: (time, type, seq, payload) in a binary heap.
    A guest takes its portions when service COMPLETES; if inventory is short it
    receives what is left (partial service) and the rest is unmet demand.
    n_guests / portions may be injected (used by deterministic verification).
    """
    A = draw_attendance(par, rng, None) if n_guests is None else n_guests
    A = int(A)
    want = (rng.choice(par.values, size=A, p=par.probs)
            if portions is None else np.asarray(portions))
    gaps = rng.exponential(1.0 / par.lam, size=A)
    arr_t = np.cumsum(gaps)

    seq = 0
    fel = []                                    # future event list
    for k in range(A):
        heapq.heappush(fel, (arr_t[k], ARRIVAL, seq, k)); seq += 1

    inv, busy, queue = int(Q), 0, []            # queue: list used as FIFO (head index)
    qhead = 0
    served = unmet = guests_short = 0
    waits = np.zeros(A)
    stockout_time = np.nan
    clock = 0.0

    def start_service(k, now):
        nonlocal busy, seq
        busy += 1
        waits[k] = now - arr_t[k]
        s = rng.exponential(1.0 / par.mu)
        heapq.heappush(fel, (now + s, DEPARTURE, seq, k)); seq += 1

    while fel:
        clock, etype, _, k = heapq.heappop(fel)
        if etype == ARRIVAL:
            if busy < par.servers:
                start_service(k, clock)
            else:
                queue.append(k)
        else:                                    # DEPARTURE: guest k finishes service
            busy -= 1
            take = min(int(want[k]), inv)
            inv -= take
            served += take
            miss = int(want[k]) - take
            unmet += miss
            if miss > 0:
                guests_short += 1
                if np.isnan(stockout_time):
                    stockout_time = clock
            if qhead < len(queue):
                start_service(queue[qhead], clock); qhead += 1

    requested = int(want.sum())
    # --- run-time conservation checks (cheap, always on) ---
    assert Q == served + inv, "prepared != served + leftover"
    assert requested == served + unmet, "requested != served + unmet"
    return {"A": A, "requested": requested, "served": served, "leftover": inv,
            "unmet": unmet, "guests_short": guests_short,
            "mean_wait": waits.mean() if A else 0.0, "max_wait": waits.max() if A else 0.0,
            "stockout_time": stockout_time, "end_time": clock}


def run_des(par, Q, reps=2_000, seed=42):
    """Independent replications of the DES; returns arrays + (mean, hw) summary."""
    rng = np.random.default_rng(seed)
    rows = [simulate_event(par, Q, rng) for _ in range(reps)]
    keys = rows[0].keys()
    arr = {k: np.array([r[k] for r in rows], float) for k in keys}
    arr["cost"] = par.c_prep * Q + par.c_short * arr["unmet"]
    arr["shortage"] = (arr["unmet"] > 0).astype(float)
    return arr


In [7]:
# 3.  M/M/c verification helper (infinite-style long run, warm-up discarded)
def mm_queue_wait(lam, mu, servers, n_cust, rng, warmup=2000):
    """Mean wait in queue of a FIFO M/M/c via the same next-event logic."""
    arr_t = np.cumsum(rng.exponential(1.0 / lam, size=n_cust))
    fel, seq = [], 0
    for k in range(n_cust):
        heapq.heappush(fel, (arr_t[k], ARRIVAL, seq, k)); seq += 1
    busy, queue, qhead = 0, [], 0
    waits = np.zeros(n_cust)
    while fel:
        t, et, _, k = heapq.heappop(fel)
        if et == ARRIVAL:
            if busy < servers:
                busy += 1; waits[k] = 0.0
                heapq.heappush(fel, (t + rng.exponential(1 / mu), DEPARTURE, seq, k)); seq += 1
            else:
                queue.append(k)
        else:
            busy -= 1
            if qhead < len(queue):
                j = queue[qhead]; qhead += 1; busy += 1
                waits[j] = t - arr_t[j]
                heapq.heappush(fel, (t + rng.exponential(1 / mu), DEPARTURE, seq, j)); seq += 1
    return waits[warmup:].mean()


In [8]:
# 3.  Stretch: multi-category menu (rice / meat / vegetables)
def draw_multi_demand(N, p, cats, rng, size):
    """Shared attendance A across categories; per-category portion distributions.
    cats: {name: (values, probs)}.  Returns {name: D_array}, A_array."""
    A = rng.binomial(N, p, size=size)
    D = {name: rng.multinomial(A, pr) @ np.array(v) for name, (v, pr) in cats.items()}
    return D, A


def optimise_demand(D, c_prep, c_short, Qgrid, alpha=0.05):
    """Grid search on a pre-drawn demand sample D (1-D array)."""
    Q = np.asarray(Qgrid)[None, :]
    unmet = np.maximum(D[:, None] - Q, 0)
    cost = (c_prep * Q + c_short * unmet).mean(axis=0)
    ps = (unmet > 0).mean(axis=0)
    feas = np.where(ps <= alpha)[0]
    i = feas[np.argmin(cost[feas])]
    return int(Qgrid[i]), float(cost[i]), float(ps[i])


## 4. Verification

Each check below is an `assert`, so a failure stops the notebook. Statistical checks use a tolerance of 3 standard errors.

| # | Check | Type |
|---|---|---|
| V1 | $E[A]=Np$, $E[D]$, $\operatorname{Var}(D)$ vs theory | analytic |
| V2 | No randomness $\Rightarrow$ demand exactly $2N$ | limiting case |
| V3 | prepared = served + leftover; requested = served + unmet (10,000 runs) | conservation |
| V4 | DES aggregates = fast-layer aggregates | independent implementations |
| V5 | M/M/1 and M/M/4 simulated $W_q$ vs closed form | analytic |
| V6 | Optimiser vs a hand-solved case | hand calculation |
| V7 | Grid-search optimum = newsvendor quantile | analytic, independent method |

In [9]:
par = Params()
se = lambda x: x.std(ddof=1) / np.sqrt(len(x))

# --- V1: Binomial attendance and demand moments -------------------------------------------
A, D = draw_demand(par, np.random.default_rng(SEED), 200_000)
print(f"V1  E[A]: sim {A.mean():.3f} vs theory {par.N*par.p:.3f}   | E[D]: sim {D.mean():.3f} vs {par.mean_demand:.3f}"
      f"   | Var D: sim {D.var():.1f} vs {par.var_demand:.1f}")
assert abs(A.mean() - par.N*par.p) < 3*se(A)
assert abs(D.mean() - par.mean_demand) < 3*se(D)
assert abs(D.var() / par.var_demand - 1) < 0.02

# --- V2: deterministic limit ----------------------------------------------------------------
pd_ = Params(N=50, p=1.0, values=(2,), probs=(1.0,))
r1 = simulate_event(pd_, 100, np.random.default_rng(SEED))
r2 = simulate_event(pd_, 90,  np.random.default_rng(SEED))
print(f"V2  deterministic: requested={r1['requested']} (expect 100), unmet={r1['unmet']}; with Q=90 unmet={r2['unmet']} (expect 10)")
assert r1["requested"] == 100 and r1["unmet"] == 0 and r1["leftover"] == 0 and r2["unmet"] == 10

# --- V3: conservation (asserted inside simulate_event on every run) -------------------------
des = run_des(par, 285, reps=10_000, seed=SEED)
assert np.all(285 == des["served"] + des["leftover"]) and np.all(des["requested"] == des["served"] + des["unmet"])
print("V3  conservation holds in all 10,000 DES runs")

# --- V4: DES vs fast layer (same model, different code paths) -------------------------------
g = evaluate_grid(par, [285], reps=200_000, seed=SEED + 1)
for k_des, k_fast in [("unmet", "unmet"), ("leftover", "unused"), ("cost", "cost")]:
    diff = abs(des[k_des].mean() - g[k_fast][0]); tol = 3 * np.hypot(se(des[k_des]), g[k_fast + "_hw"][0] / 1.96)
    print(f"V4  {k_des:9s}: DES {des[k_des].mean():9.2f}  fast {g[k_fast][0]:9.2f}  |diff|={diff:.2f} <= {tol:.2f}")
    assert diff <= tol
pdiff = abs(des["shortage"].mean() - g["p_short"][0]); ptol = 3 * np.hypot(se(des["shortage"]), g["p_short_hw"][0] / 1.96)
print(f"V4  P(short) : DES {des['shortage'].mean():.4f}  fast {g['p_short'][0]:.4f}"); assert pdiff <= ptol

# --- V5: queue waits vs closed forms --------------------------------------------------------
def erlang_c_wq(lam, mu, c):
    a, rho = lam / mu, lam / (c * mu)
    top = a**c / factorial(c) / (1 - rho)
    return (top / (sum(a**k / factorial(k) for k in range(c)) + top)) / (c * mu - lam)

for lam, mu, c, n in [(6, 8, 1, 40_000), (par.lam, par.mu, par.servers, 40_000)]:
    w = np.array([mm_queue_wait(lam, mu, c, n, np.random.default_rng(SEED + s)) for s in range(20)])
    th = erlang_c_wq(lam, mu, c)
    print(f"V5  M/M/{c} (lam={lam}, mu={mu}, rho={lam/(c*mu):.2f}): Wq sim {w.mean():.4f} +/- {1.96*se(w):.4f}  theory {th:.4f}")
    assert abs(w.mean() - th) < 3 * se(w)
assert abs(erlang_c_wq(6, 8, 1) - 6 / (8 * (8 - 6))) < 1e-12        # Erlang-C reduces to M/M/1

# --- V6: hand-solved optimisation case ------------------------------------------------------
# N=4, p=.5, 1 portion each => D ~ Bin(4,.5), pmf (1,4,6,4,1)/16. c_prep=1, c_short=4.
# By hand: E[C(2)]=3.5, E[C(3)]=3.25, E[C(4)]=4.  Unconstrained Q=3. P(D>3)=1/16=.0625>.05 => constrained Q=4.
pt = Params(N=4, p=0.5, values=(1,), probs=(1.0,), c_prep=1.0, c_short=4.0)
o = optimise(pt, np.arange(0, 7), reps=400_000, seed=SEED)
g_ = o["grid"]
print(f"V6  grid search: Q_unc={o['Q_unc']} (hand 3), cost {g_['cost'][3]:.3f} (hand 3.250);  Q_opt={o['Q_opt']} (hand 4), cost {g_['cost'][4]:.3f} (hand 4.000)")
assert o["Q_unc"] == 3 and o["Q_opt"] == 4
assert abs(g_["cost"][3] - 3.25) < 3 * g_["cost_hw"][3] / 1.96 and abs(g_["cost"][2] - 3.5) < 3 * g_["cost_hw"][2] / 1.96

# --- V7: grid-search optimum vs newsvendor critical-fractile quantile ------------------------
o7 = optimise(par, np.arange(230, 340), reps=10_000, seed=SEED)
_, Dbig = draw_demand(par, np.random.default_rng(SEED + 2), 1_000_000)
q_nv = np.quantile(Dbig, 1 - par.c_prep / par.c_short, method="higher")
print(f"V7  unconstrained grid optimum {o7['Q_unc']}  vs newsvendor quantile (fractile {1-par.c_prep/par.c_short:.2f}) {q_nv:.0f}")
assert abs(o7["Q_unc"] - q_nv) <= 1
print("\nALL VERIFICATION CHECKS PASSED")

V1  E[A]: sim 180.000 vs theory 180.000   | E[D]: sim 270.022 vs 270.000   | Var D: sim 139.6 vs 139.5
V2  deterministic: requested=100 (expect 100), unmet=0; with Q=90 unmet=10 (expect 10)
V3  conservation holds in all 10,000 DES runs
V4  unmet    : DES      0.55  fast      0.57  |diff|=0.02 <= 0.07
V4  leftover : DES     15.58  fast     15.59  |diff|=0.00 <= 0.33
V4  cost     : DES  86329.65  fast  86357.41  |diff|=27.76 <= 103.49
V4  P(short) : DES 0.0928  fast 0.0946
V5  M/M/1 (lam=6, mu=8, rho=0.75): Wq sim 0.3735 +/- 0.0071  theory 0.3750
V5  M/M/4 (lam=5.0, mu=1.5, rho=0.83): Wq sim 0.6596 +/- 0.0192  theory 0.6577
V6  grid search: Q_unc=3 (hand 3), cost 3.249 (hand 3.250);  Q_opt=4 (hand 4), cost 4.000 (hand 4.000)
V7  unconstrained grid optimum 280  vs newsvendor quantile (fractile 0.80) 280

ALL VERIFICATION CHECKS PASSED
